这份代码对单个音频D:\Project_Github\Indo-Pacific-humpback-dolphin\09_Locate\Voyage_Ori_10\PulseTrain_291\Click_007\Pulse_007.wav进行特征提取，基本能够跑通

In [4]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
单个 click 音频的显式多径结构提取
--------------------------------
输入：
    一个已截取好的单个 click WAV，直达/主到达大致位于音频中间。

输出：
    1. selected_paths.csv         主到达 + 最多两个多径
    2. all_candidates.csv         所有候选及其评分，便于调参
    3. multipath_detection.png    原始波形、局部 RMS 与选中位置
    4. path_waveform_overlay.png  主到达和多径片段的归一化波形叠加
    5. run_summary.txt            本次运行参数与简要结果

说明：
    - 不进行 RAE、去混响或复杂降噪。
    - 先在音频中部寻找局部 RMS 最大的“主到达”。
    - 再在主到达之后 4 ms 内产生候选。
    - 使用能量、与主到达的归一化互相关、频谱相似度进行受约束贪婪选择。
    - 最多保留两个高置信度多径；不强制补齐。
"""

from __future__ import annotations

import argparse
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.io import wavfile
from scipy.signal import correlate, find_peaks, hilbert


# ============================================================
# 1. 用户主要需要修改的路径
# ============================================================
DEFAULT_WAV_PATH = r"D:\Project_Github\Indo-Pacific-humpback-dolphin\09_Locate\Voyage_Ori_10\PulseTrain_291\Click_007\Pulse_007.wav"
DEFAULT_OUTPUT_DIR = r"multipath_results"


# ============================================================
# 2. 核心参数
# ============================================================
SEGMENT_US = 50.0                 # 每条到达成分的固定片段长度
MAX_DELAY_MS = 4.0                # 主到达之后的最大多径搜索时延
MAX_MULTIPATHS = 2                # 最多保留的多径数量

CENTER_SEARCH_HALF_WIDTH_MS = 1.5 # 在音频中心 ±1.5 ms 内寻找主到达
DIRECT_GUARD_US = 100.0           # 主到达后留出保护区，避免将自身振铃重复选中
CANDIDATE_MIN_DISTANCE_US = 50.0  # 候选峰之间的最小间隔

# 候选门限
PEAK_HEIGHT_MAD_FACTOR = 3.0      # 局部 RMS 峰高度门限：median + factor * MAD
PEAK_PROMINENCE_MAD_FACTOR = 1.5  # 峰显著性门限
MIN_CANDIDATE_SNR_DB = 2.0        # 候选相对稳健噪声底的最低 SNR
MIN_ABS_NCC = 0.20                # 与主到达的最低绝对归一化互相关
MIN_SPECTRAL_COSINE = 0.20        # 与主到达的最低频谱余弦相似度
MIN_ACCEPT_SCORE = 0.35           # 贪婪选择的最低综合评分

# 综合评分权重
WEIGHT_ENERGY = 0.40
WEIGHT_NCC = 0.40
WEIGHT_SPECTRAL = 0.20


@dataclass
class PathResult:
    label: str
    center_sample: int
    center_time_ms: float
    relative_delay_ms: float
    start_sample: int
    end_sample: int
    rms: float
    relative_rms_db: float
    abs_ncc: float
    spectral_cosine: float
    snr_db: float
    score: float


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="从单个 click WAV 中提取主到达和最多两个多径成分。"
    )
    parser.add_argument(
        "--wav",
        type=str,
        default=DEFAULT_WAV_PATH,
        help="输入 WAV 路径；不提供时使用脚本顶部 DEFAULT_WAV_PATH。",
    )
    parser.add_argument(
        "--output",
        type=str,
        default=DEFAULT_OUTPUT_DIR,
        help="输出目录。",
    )
    # parse_known_args 可忽略 Jupyter/IPython 内核传递的额外参数
    args, _ = parser.parse_known_args()
    return args


def load_wav_mono(path: Path) -> Tuple[int, np.ndarray]:
    if not path.exists():
        raise FileNotFoundError(f"找不到 WAV 文件：{path}")

    fs, data = wavfile.read(path)

    if data.ndim == 2:
        data = data.astype(np.float64).mean(axis=1)

    original_dtype = data.dtype
    data = data.astype(np.float64)

    if np.issubdtype(original_dtype, np.integer):
        info = np.iinfo(original_dtype)
        scale = max(abs(info.min), abs(info.max))
        data /= float(scale)
    elif np.issubdtype(original_dtype, np.floating):
        peak = np.max(np.abs(data))
        if peak > 1.0:
            data /= peak
    else:
        raise TypeError(f"不支持的 WAV 数据类型：{original_dtype}")

    if not np.all(np.isfinite(data)):
        raise ValueError("音频包含 NaN 或无穷值。")

    data -= np.mean(data)

    peak = np.max(np.abs(data))
    if peak <= np.finfo(float).eps:
        raise ValueError("音频几乎为全零，无法提取多径。")

    data /= peak
    return int(fs), data


def ms_to_samples(ms: float, fs: int) -> int:
    return max(1, int(round(ms * 1e-3 * fs)))


def us_to_samples(us: float, fs: int) -> int:
    return max(1, int(round(us * 1e-6 * fs)))


def moving_rms(x: np.ndarray, window: int) -> np.ndarray:
    kernel = np.ones(window, dtype=np.float64) / window
    return np.sqrt(np.convolve(x * x, kernel, mode="same"))


def robust_median_mad(values: np.ndarray) -> Tuple[float, float]:
    values = np.asarray(values, dtype=np.float64)
    median = float(np.median(values))
    mad = float(np.median(np.abs(values - median)))
    robust_sigma = 1.4826 * mad
    return median, max(robust_sigma, np.finfo(float).eps)


def extract_centered_segment(
    x: np.ndarray, center: int, length: int
) -> Tuple[np.ndarray, int, int]:
    half_left = length // 2
    start = center - half_left
    end = start + length

    if start < 0 or end > len(x):
        raise ValueError(
            f"候选片段越界：center={center}, start={start}, end={end}, len={len(x)}"
        )
    return x[start:end].copy(), start, end


def rms_value(x: np.ndarray) -> float:
    return float(np.sqrt(np.mean(np.square(x))))


def max_abs_normalized_cross_correlation(
    reference: np.ndarray, candidate: np.ndarray
) -> float:
    ref = reference - np.mean(reference)
    cand = candidate - np.mean(candidate)

    denom = np.linalg.norm(ref) * np.linalg.norm(cand)
    if denom <= np.finfo(float).eps:
        return 0.0

    corr = correlate(cand, ref, mode="full", method="auto") / denom
    return float(np.max(np.abs(corr)))


def spectral_cosine_similarity(
    reference: np.ndarray, candidate: np.ndarray
) -> float:
    window = np.hanning(len(reference))
    ref_spec = np.abs(np.fft.rfft(reference * window))
    cand_spec = np.abs(np.fft.rfft(candidate * window))

    denom = np.linalg.norm(ref_spec) * np.linalg.norm(cand_spec)
    if denom <= np.finfo(float).eps:
        return 0.0

    value = float(np.dot(ref_spec, cand_spec) / denom)
    return float(np.clip(value, 0.0, 1.0))


def estimate_noise_rms(
    rms_curve: np.ndarray,
    direct_center: int,
    exclusion_samples: int,
) -> float:
    mask = np.ones(len(rms_curve), dtype=bool)
    start = max(0, direct_center - exclusion_samples)
    end = min(len(rms_curve), direct_center + exclusion_samples + 1)
    mask[start:end] = False

    background = rms_curve[mask]
    if background.size == 0:
        background = rms_curve

    median, robust_sigma = robust_median_mad(background)
    return max(median + robust_sigma, np.finfo(float).eps)


def find_direct_arrival(
    rms_curve: np.ndarray,
    fs: int,
    segment_len: int,
) -> int:
    n = len(rms_curve)
    midpoint = n // 2
    half_width = ms_to_samples(CENTER_SEARCH_HALF_WIDTH_MS, fs)

    lower = max(segment_len // 2, midpoint - half_width)
    upper = min(n - (segment_len - segment_len // 2), midpoint + half_width)

    if lower >= upper:
        raise ValueError(
            "音频过短，无法在中心搜索窗口中提取固定长度主到达片段。"
        )

    return int(lower + np.argmax(rms_curve[lower:upper]))


def generate_candidates(
    x: np.ndarray,
    rms_curve: np.ndarray,
    fs: int,
    direct_center: int,
    direct_segment: np.ndarray,
    segment_len: int,
) -> pd.DataFrame:
    guard_samples = us_to_samples(DIRECT_GUARD_US, fs)
    max_delay_samples = ms_to_samples(MAX_DELAY_MS, fs)
    min_peak_distance = us_to_samples(CANDIDATE_MIN_DISTANCE_US, fs)

    left_margin = segment_len // 2
    right_margin = segment_len - left_margin

    search_start = max(
        direct_center + guard_samples,
        left_margin,
    )
    search_end = min(
        direct_center + max_delay_samples,
        len(x) - right_margin,
    )

    if search_start >= search_end:
        raise ValueError(
            "主到达到音频末尾的有效长度不足 4 ms。"
            "请增加输入片段长度，或减小 MAX_DELAY_MS。"
        )

    search_rms = rms_curve[search_start:search_end + 1]
    median, robust_sigma = robust_median_mad(search_rms)

    height_threshold = median + PEAK_HEIGHT_MAD_FACTOR * robust_sigma
    prominence_threshold = PEAK_PROMINENCE_MAD_FACTOR * robust_sigma

    local_peaks, properties = find_peaks(
        search_rms,
        height=height_threshold,
        prominence=prominence_threshold,
        distance=max(1, min_peak_distance),
    )

    # 当稳健门限没有产生候选时，保留搜索区内最强局部峰作为调试候选，
    # 但后续仍需通过 SNR、相关性和综合评分门限。
    if local_peaks.size == 0:
        fallback_peaks, fallback_properties = find_peaks(
            search_rms,
            distance=max(1, min_peak_distance),
        )
        if fallback_peaks.size > 0:
            order = np.argsort(search_rms[fallback_peaks])[::-1]
            local_peaks = fallback_peaks[order[: min(10, len(order))]]
            properties = fallback_properties

    direct_rms = rms_value(direct_segment)
    noise_rms = estimate_noise_rms(
        rms_curve,
        direct_center,
        exclusion_samples=max(ms_to_samples(MAX_DELAY_MS, fs), segment_len),
    )

    rows = []
    for local_idx in local_peaks:
        center = int(search_start + local_idx)

        try:
            segment, start, end = extract_centered_segment(
                x, center, segment_len
            )
        except ValueError:
            continue

        candidate_rms = rms_value(segment)
        relative_rms_db = 20.0 * np.log10(
            max(candidate_rms, np.finfo(float).eps)
            / max(direct_rms, np.finfo(float).eps)
        )
        snr_db = 20.0 * np.log10(
            max(candidate_rms, np.finfo(float).eps) / noise_rms
        )

        abs_ncc = max_abs_normalized_cross_correlation(
            direct_segment, segment
        )
        spectral_cosine = spectral_cosine_similarity(
            direct_segment, segment
        )

        # 能量项相对主到达归一化；比主到达更强时封顶为 1。
        energy_score = float(
            np.clip(candidate_rms / max(direct_rms, np.finfo(float).eps), 0.0, 1.0)
        )

        score = (
            WEIGHT_ENERGY * energy_score
            + WEIGHT_NCC * abs_ncc
            + WEIGHT_SPECTRAL * spectral_cosine
        )

        passes_thresholds = (
            snr_db >= MIN_CANDIDATE_SNR_DB
            and abs_ncc >= MIN_ABS_NCC
            and spectral_cosine >= MIN_SPECTRAL_COSINE
            and score >= MIN_ACCEPT_SCORE
        )

        rows.append(
            {
                "center_sample": center,
                "center_time_ms": 1000.0 * center / fs,
                "relative_delay_ms": 1000.0 * (center - direct_center) / fs,
                "start_sample": start,
                "end_sample": end,
                "rms": candidate_rms,
                "relative_rms_db": relative_rms_db,
                "abs_ncc": abs_ncc,
                "spectral_cosine": spectral_cosine,
                "snr_db": snr_db,
                "score": score,
                "passes_thresholds": bool(passes_thresholds),
            }
        )

    columns = [
        "center_sample",
        "center_time_ms",
        "relative_delay_ms",
        "start_sample",
        "end_sample",
        "rms",
        "relative_rms_db",
        "abs_ncc",
        "spectral_cosine",
        "snr_db",
        "score",
        "passes_thresholds",
    ]
    return pd.DataFrame(rows, columns=columns)


def greedy_select_candidates(
    candidates: pd.DataFrame,
    segment_len: int,
) -> pd.DataFrame:
    if candidates.empty:
        return candidates.copy()

    eligible = candidates[candidates["passes_thresholds"]].copy()
    eligible = eligible.sort_values("score", ascending=False)

    selected_rows = []
    selected_centers: List[int] = []

    # 避免两个选择片段高度重叠。
    minimum_separation = segment_len

    for _, row in eligible.iterrows():
        center = int(row["center_sample"])
        if any(abs(center - previous) < minimum_separation for previous in selected_centers):
            continue

        selected_rows.append(row)
        selected_centers.append(center)

        if len(selected_rows) >= MAX_MULTIPATHS:
            break

    if not selected_rows:
        return eligible.iloc[0:0].copy()

    selected = pd.DataFrame(selected_rows)
    selected = selected.sort_values("relative_delay_ms").reset_index(drop=True)
    return selected


def build_path_table(
    x: np.ndarray,
    fs: int,
    direct_center: int,
    direct_segment: np.ndarray,
    direct_start: int,
    direct_end: int,
    selected: pd.DataFrame,
    noise_rms: float,
) -> pd.DataFrame:
    direct_rms = rms_value(direct_segment)
    direct_snr_db = 20.0 * np.log10(
        max(direct_rms, np.finfo(float).eps) / noise_rms
    )

    results = [
        PathResult(
            label="main_arrival",
            center_sample=direct_center,
            center_time_ms=1000.0 * direct_center / fs,
            relative_delay_ms=0.0,
            start_sample=direct_start,
            end_sample=direct_end,
            rms=direct_rms,
            relative_rms_db=0.0,
            abs_ncc=1.0,
            spectral_cosine=1.0,
            snr_db=direct_snr_db,
            score=1.0,
        )
    ]

    for index, row in selected.iterrows():
        results.append(
            PathResult(
                label=f"multipath_{index + 1}",
                center_sample=int(row["center_sample"]),
                center_time_ms=float(row["center_time_ms"]),
                relative_delay_ms=float(row["relative_delay_ms"]),
                start_sample=int(row["start_sample"]),
                end_sample=int(row["end_sample"]),
                rms=float(row["rms"]),
                relative_rms_db=float(row["relative_rms_db"]),
                abs_ncc=float(row["abs_ncc"]),
                spectral_cosine=float(row["spectral_cosine"]),
                snr_db=float(row["snr_db"]),
                score=float(row["score"]),
            )
        )

    return pd.DataFrame([asdict(item) for item in results])


def save_detection_plot(
    x: np.ndarray,
    rms_curve: np.ndarray,
    fs: int,
    paths: pd.DataFrame,
    output_path: Path,
) -> None:
    time_ms = np.arange(len(x)) * 1000.0 / fs

    scaled_rms = rms_curve / max(np.max(rms_curve), np.finfo(float).eps)
    waveform_scale = max(np.max(np.abs(x)), np.finfo(float).eps)
    scaled_rms = scaled_rms * waveform_scale

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.plot(time_ms, x, linewidth=0.8, label="Waveform")
    ax.plot(time_ms, scaled_rms, linewidth=1.0, alpha=0.8, label="Scaled local RMS")

    for _, row in paths.iterrows():
        start_ms = 1000.0 * int(row["start_sample"]) / fs
        end_ms = 1000.0 * int(row["end_sample"]) / fs
        center_ms = float(row["center_time_ms"])
        label = str(row["label"])

        ax.axvspan(start_ms, end_ms, alpha=0.16)
        ax.axvline(center_ms, linewidth=1.0, linestyle="--")
        ax.text(
            center_ms,
            0.92 * waveform_scale,
            label,
            rotation=90,
            va="top",
            ha="right",
            fontsize=9,
        )

    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Normalized amplitude")
    ax.set_title("Explicit main-arrival and multipath extraction")
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")
    fig.tight_layout()
    fig.savefig(output_path, dpi=220)
    plt.close(fig)


def save_overlay_plot(
    x: np.ndarray,
    fs: int,
    paths: pd.DataFrame,
    output_path: Path,
) -> None:
    fig, ax = plt.subplots(figsize=(9, 5))

    for _, row in paths.iterrows():
        start = int(row["start_sample"])
        end = int(row["end_sample"])
        segment = x[start:end].copy()
        segment -= np.mean(segment)

        scale = np.max(np.abs(segment))
        if scale > np.finfo(float).eps:
            segment /= scale

        time_us = np.arange(len(segment)) * 1e6 / fs
        ax.plot(time_us, segment, linewidth=1.1, label=str(row["label"]))

    ax.set_xlabel("Time within segment (μs)")
    ax.set_ylabel("Peak-normalized amplitude")
    ax.set_title("Waveform comparison of selected arrivals")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(output_path, dpi=220)
    plt.close(fig)


def write_summary(
    path: Path,
    wav_path: Path,
    fs: int,
    duration_ms: float,
    segment_len: int,
    paths: pd.DataFrame,
    candidate_count: int,
) -> None:
    lines = [
        f"Input WAV: {wav_path}",
        f"Sampling rate: {fs} Hz",
        f"Audio duration: {duration_ms:.6f} ms",
        f"Segment length: {SEGMENT_US:.3f} μs ({segment_len} samples)",
        f"Maximum multipath delay: {MAX_DELAY_MS:.3f} ms",
        f"Candidate count: {candidate_count}",
        f"Selected multipath count: {max(0, len(paths) - 1)}",
        "",
        "Selected arrivals:",
    ]

    for _, row in paths.iterrows():
        lines.append(
            f"- {row['label']}: "
            f"time={row['center_time_ms']:.6f} ms, "
            f"delay={row['relative_delay_ms']:.6f} ms, "
            f"relative_RMS={row['relative_rms_db']:.3f} dB, "
            f"|NCC|={row['abs_ncc']:.3f}, "
            f"spectral_cosine={row['spectral_cosine']:.3f}, "
            f"score={row['score']:.3f}"
        )

    path.write_text("\n".join(lines), encoding="utf-8")


def main() -> int:
    args = parse_args()
    wav_path = Path(args.wav).expanduser()
    output_dir = Path(args.output).expanduser()
    output_dir.mkdir(parents=True, exist_ok=True)

    if not wav_path.exists():
        print(
            f"找不到 WAV 文件：{wav_path}\n"
            "请修改脚本顶部 DEFAULT_WAV_PATH，"
            "或运行时使用：python multipath_structure_extraction.py --wav 你的音频.wav",
            file=sys.stderr,
        )
        return 2

    fs, x = load_wav_mono(wav_path)
    duration_ms = 1000.0 * len(x) / fs

    segment_len = us_to_samples(SEGMENT_US, fs)
    if segment_len < 3:
        raise ValueError(
            f"采样率 {fs} Hz 下，50 μs 仅对应 {segment_len} 个采样点，"
            "采样率过低，无法可靠提取该长度片段。"
        )

    required_duration_ms = (
        CENTER_SEARCH_HALF_WIDTH_MS + MAX_DELAY_MS + SEGMENT_US / 1000.0
    )
    if duration_ms < required_duration_ms:
        raise ValueError(
            f"音频时长仅 {duration_ms:.3f} ms，可能不足以覆盖中间主到达之后 "
            f"{MAX_DELAY_MS:.3f} ms 的搜索范围。"
        )

    rms_curve = moving_rms(x, segment_len)
    direct_center = find_direct_arrival(rms_curve, fs, segment_len)
    direct_segment, direct_start, direct_end = extract_centered_segment(
        x, direct_center, segment_len
    )

    candidates = generate_candidates(
        x=x,
        rms_curve=rms_curve,
        fs=fs,
        direct_center=direct_center,
        direct_segment=direct_segment,
        segment_len=segment_len,
    )
    selected = greedy_select_candidates(candidates, segment_len)

    noise_rms = estimate_noise_rms(
        rms_curve,
        direct_center,
        exclusion_samples=max(ms_to_samples(MAX_DELAY_MS, fs), segment_len),
    )

    paths = build_path_table(
        x=x,
        fs=fs,
        direct_center=direct_center,
        direct_segment=direct_segment,
        direct_start=direct_start,
        direct_end=direct_end,
        selected=selected,
        noise_rms=noise_rms,
    )

    candidates.to_csv(
        output_dir / "all_candidates.csv",
        index=False,
        encoding="utf-8-sig",
    )
    paths.to_csv(
        output_dir / "selected_paths.csv",
        index=False,
        encoding="utf-8-sig",
    )

    save_detection_plot(
        x=x,
        rms_curve=rms_curve,
        fs=fs,
        paths=paths,
        output_path=output_dir / "multipath_detection.png",
    )
    save_overlay_plot(
        x=x,
        fs=fs,
        paths=paths,
        output_path=output_dir / "path_waveform_overlay.png",
    )
    write_summary(
        path=output_dir / "run_summary.txt",
        wav_path=wav_path,
        fs=fs,
        duration_ms=duration_ms,
        segment_len=segment_len,
        paths=paths,
        candidate_count=len(candidates),
    )

    print(f"处理完成：{wav_path}")
    print(f"采样率：{fs} Hz")
    print(f"音频时长：{duration_ms:.3f} ms")
    print(f"主到达位置：{paths.iloc[0]['center_time_ms']:.6f} ms")
    print(f"选中多径数量：{max(0, len(paths) - 1)}")
    print(f"结果目录：{output_dir.resolve()}")

    if len(paths) == 1:
        print(
            "未选出满足当前门限的多径。"
            "请优先查看 all_candidates.csv，再适当降低 "
            "MIN_ABS_NCC、MIN_CANDIDATE_SNR_DB 或 MIN_ACCEPT_SCORE。"
        )

    return 0


if __name__ == "__main__":
    try:
        sys.exit(main())
    except SystemExit as e:
        if e.code != 0:
            print(f"\n程序退出，返回码: {e.code}")


处理完成：D:\Project_Github\Indo-Pacific-humpback-dolphin\09_Locate\Voyage_Ori_10\PulseTrain_291\Click_007\Pulse_007.wav
采样率：576000 Hz
音频时长：10.002 ms
主到达位置：5.019097 ms
选中多径数量：2
结果目录：D:\Project_Github\Indo-Pacific-humpback-dolphin\17_20260805\multipath_results
